# 第十一章：轨迹与拟时序分析（PAGA + DPT + CytoTRACE2）

**scRNA-seq 实践教程系列** | 基于 GSE136103 肝硬化数据集

---

## 写在前面

前面的章节中，我们一直在"静态"地观察细胞——数它们有多少个、看它们表达了什么基因、分析它们之间的通讯。但细胞是动态的：单核细胞会分化成巨噬细胞、naive T 细胞会被激活变成 effector T 细胞。

轨迹分析（trajectory analysis）和拟时序分析（pseudotime analysis） 试图从 scRNA-seq 的"快照"数据中重建这些动态过程。核心思想是：如果一群细胞正在经历同一个分化过程，那么不同细胞恰好被"冻结"在了不同阶段——通过计算它们在表达空间中的"距离"，可以把它们排列成一条连续的轨迹。

这一篇我们将使用两个经典方法：

PAGA（Partition-based Graph Abstraction）：全局拓扑，看细胞类型之间的连接关系
Diffusion Pseudotime（DPT）：局部轨迹，给每个细胞一个"分化进度条"

## 为什么不是真正的"时间"

在开始之前，必须先澄清一个根本性的概念：

⚠️ 踩坑预警：拟时序 ≠ 真实时间

scRNA-seq 是在某一个时间点对所有细胞做的测量——它是一张快照，不是一段延时视频。拟时序（pseudotime）是一个计算推断：假设细胞沿着某条轨迹连续变化，那么每个细胞应该处于这条轨迹的哪个位置？

这个推断有两个根本限制：

不能证明方向：如果我们看到 A → B → C 的轨迹，不能排除实际是 C → B → A。方向需要额外的生物学证据（比如已知的分化方向）。
不能证明因果：两种细胞在轨迹上相邻，不代表一种真的会"变成"另一种。它们可能只是转录组相似。

真正的谱系追踪需要 lineage tracing（基于条形码、CRISPR scarring 等实验技术）。拟时序是它的计算替代品——便宜但不精确。

## Part A：PAGA——全局拓扑分析

### 什么是 PAGA

PAGA（Wolf et al., Genome Biology, 2019）通过比较 cluster 之间的实际连接强度和随机期望的连接强度，来判断哪些细胞类型之间存在真实的"过渡态"连接。

直觉上：如果两个 cluster 之间有大量细胞在 k-NN 图上互相连接，那它们之间可能存在连续的分化关系。

### 运行 PAGA

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

adata = sc.read_h5ad("results/05_annotated.h5ad")

# PAGA 需要在已有的 neighbor graph 上运行
sc.tl.paga(adata, groups="cell_type")

# 导出连接矩阵
paga_conn = pd.DataFrame(
    adata.uns["paga"]["connectivities"].toarray(),
    index=adata.obs["cell_type"].cat.categories,
    columns=adata.obs["cell_type"].cat.categories
)
paga_conn.to_csv("results/paga_connectivity.csv")
print("PAGA connectivity (top connections):")
print(paga_conn.stack()
      .sort_values(ascending=False)
      .head(10).to_string())

**输出：**

> 📊 输出：
> PAGA connectivity (top connections):
> T cells      - NK cells        0.8421
> Monocytes    - Macrophages     0.7356
> Monocytes    - cDC             0.5012
> Macrophages  - cDC             0.4278
> Endothelial  - HSCs/Mesenchyme 0.2134
> NK cells     - Monocytes       0.1587
> B cells      - Plasma cells    0.1203
> T cells      - Monocytes       0.0845
> Hepatocytes  - Endothelial     0.0632
> cDC          - pDC             0.0498

### PAGA 可视化

In [ ]:
sc.pl.paga(
    adata,
    threshold=0.05,    # 只显示连接强度 > 0.05 的边
    node_size_scale=2,
    edge_width_scale=1.5,
    fontsize=8,
    show=False
)
plt.savefig(
    "results/figures/10_paga_all_celltypes.png",
    dpi=150, bbox_inches="tight"
)
plt.close()

图 1：PAGA 图——细胞类型间的连接关系

图 1：PAGA 全局拓扑图。 每个节点代表一种细胞类型，节点大小正比于细胞数量，边的粗细正比于连接强度。可以清楚地看到：

Monocytes ↔ Macrophages：强连接（0.74），反映了单核-巨噬细胞分化轴——这是我们最关心的轨迹
T cells ↔ NK cells：强连接（0.84），反映了它们共享的淋巴系起源和部分重叠的转录组
B cells ↔ Plasma cells：中等连接（0.12），B 细胞到浆细胞的分化
Endothelial ↔ HSCs：弱连接（0.21），可能反映了内皮-间充质转换（EndMT）或空间邻近性

💡 Tip：PAGA 连接 ≠ 分化关系

PAGA 图显示的是转录组空间中的"可达性"——两个 cluster 之间有连接，可能是因为真实的分化关系（Monocytes → Macrophages），也可能只是因为它们的表达谱相似（T cells ↔ NK cells 共享 NKG7 等基因）。解读时需要结合已有的生物学知识。

## Part B：Myeloid DPT——单核细胞到巨噬细胞的分化轨迹

### 为什么聚焦髓系细胞

Ramachandran 2019 的核心发现是：肝硬化中单核细胞从外周血浸润肝脏，分化为 TREM2+CD9+ 的瘢痕相关巨噬细胞（SAMac）。如果这条分化轨迹是真的，我们应该能在拟时序分析中看到从 Monocytes 到 LAM/SAMac 的连续过渡，伴随 S100A8/S100A9 的下调和 C1QA/TREM2 的上调。

### 提取髓系细胞 + 计算 DPT

In [ ]:
# 提取 Monocytes + Macrophages
myeloid_types = [
    "CD14 Mono", "CD16 Mono",
    "Kupffer cells", "LAM",
    "Resident Mac", "Inflammatory Mac"
]
adata_mye = adata[
    adata.obs["cell_type_fine"].isin(myeloid_types)
].copy()
print(f"Myeloid cells: {adata_mye.n_obs}")

**输出：**

> 📊 输出：
> Myeloid cells: 8273

In [ ]:
# 重新做降维（子集数据需要重新计算 neighbor graph）
sc.pp.neighbors(
    adata_mye, use_rep="X_pca_harmony",
    n_neighbors=15, n_pcs=20
)
sc.tl.umap(adata_mye, min_dist=0.3)
sc.tl.diffmap(adata_mye, n_comps=10)

### 选择 root cell：生物学先验最重要

DPT 的结果完全取决于你选择哪个细胞作为起点。这不是算法能自动解决的——需要你的生物学判断。

我们知道分化方向是 Monocyte → Macrophage。S100A8 是单核细胞的标志基因，在分化过程中逐渐下调。所以我们选 S100A8 表达最高的单核细胞 作为 root：

In [ ]:
# 在 CD14 Mono 中找 S100A8 表达最高的细胞
mono_mask = (
    adata_mye.obs["cell_type_fine"] == "CD14 Mono"
)
s100a8_expr = adata_mye[mono_mask, "S100A8"].X
if hasattr(s100a8_expr, "toarray"):
    s100a8_expr = s100a8_expr.toarray()
root_idx = np.where(mono_mask)[0][
    np.argmax(s100a8_expr.flatten())
]
adata_mye.uns["iroot"] = root_idx
print(f"Root cell index: {root_idx}")
print(f"Root cell type: "
      f"{adata_mye.obs['cell_type_fine'].iloc[root_idx]}")

**输出：**

> 📊 输出：
> Root cell index: 1247
> Root cell type: CD14 Mono

⚠️ 踩坑预警：Root cell 的选择会根本性地改变 DPT 结果

如果你把 root 设在 Kupffer 细胞上，整个拟时序就会反转——变成"从巨噬细胞出发"。如果设在 LAM 上，会得到一条从 SAMac 向两端分化的轨迹，这在生物学上没有意义。

经验法则： root cell 应该是你认为"最原始/未分化"的细胞。对于髓系，是外周血来的 CD14+ 单核细胞。对于 T 细胞，是 naive T 细胞。

### 计算 DPT

In [ ]:
sc.tl.dpt(adata_mye, n_dcs=10)
print(adata_mye.obs["dpt_pseudotime"].describe())

**输出：**

> 📊 输出：
> count    8273.000000
> mean        0.4521
> std         0.2387
> min         0.0000
> 25%         0.2634
> 50%         0.4312
> 75%         0.6445
> max         1.0000

### 拟时序 UMAP + 基因趋势

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左图：按细胞类型着色
sc.pl.umap(
    adata_mye, color="cell_type_fine",
    ax=axes[0], show=False,
    title="Myeloid Subtypes",
    frameon=False, legend_fontsize=7
)

# 右图：按拟时序着色
sc.pl.umap(
    adata_mye, color="dpt_pseudotime",
    ax=axes[1], show=False,
    title="Diffusion Pseudotime",
    frameon=False, cmap="viridis"
)

plt.tight_layout()
plt.savefig(
    "results/figures/10_myeloid_dpt_umap.png",
    dpi=150, bbox_inches="tight"
)
plt.close()

图 2：髓系细胞 Diffusion Pseudotime

图 2：髓系细胞 DPT 结果。 左图按细胞亚型着色，右图按拟时序着色（深色=早期/单核细胞端，亮色=晚期/巨噬细胞端）。如果分化轨迹是真实的，应该看到颜色从 CD14 Mono 区域逐渐过渡到 Kupffer/LAM 区域。

### 基因表达沿拟时序的变化趋势

这是轨迹分析中最有价值的图——看关键基因如何随拟时序变化：

In [ ]:
trajectory_genes = [
    "S100A8", "S100A9",  # 单核细胞 marker（应下降）
    "C1QA", "C1QB",      # 巨噬细胞 marker（应上升）
    "TREM2", "CD9",      # SAMac marker（应在终末端上升）
    "MARCO", "CD163"     # Kupffer marker
]
trajectory_genes = [
    g for g in trajectory_genes
    if g in adata_mye.raw.var_names
]

sc.pl.draw_graph  # 不实际运行，用 heatmap 替代
# 按拟时序排列的基因表达热图
sc.pl.heatmap(
    adata_mye,
    var_names=trajectory_genes,
    groupby="cell_type_fine",
    use_raw=True,
    show=False
)
plt.savefig(
    "results/figures/10_myeloid_gene_trends.png",
    dpi=150, bbox_inches="tight"
)
plt.close()

图 3：髓系细胞沿拟时序的基因表达变化

关键发现：

S100A8 / S100A9：在 CD14 Mono 中高表达，沿拟时序逐渐下降 ✅
C1QA / C1QB：在 Kupffer 和 Resident Mac 中高表达，沿拟时序逐渐上升 ✅
TREM2：主要在 LAM 中表达，在拟时序终末端上升 ✅

这些趋势和 Ramachandran 2019 报告的 monocyte-to-SAMac 分化轨迹高度一致。

### 保存结果

In [ ]:
myeloid_dpt = adata_mye.obs[[
    "cell_type_fine", "condition", "donor",
    "dpt_pseudotime"
]].copy()
myeloid_dpt.to_csv("results/myeloid_dpt.csv")
print("✅ Myeloid DPT saved")

## Part C：T 细胞 DPT——从 naive 到 effector

### T 细胞轨迹的生物学背景

T 细胞从 naive 状态经过抗原刺激，分化为 effector/memory 状态。在肝脏这样的免疫器官中，我们预期能看到这条轨迹。root 自然选在 CD4 naive T 细胞——它们是最"原始"的 T 细胞状态。

In [ ]:
# 提取 T 细胞
t_types = [
    "CD4 naive", "CD4 memory", "Treg",
    "CD8 memory", "CD8 effector",
    "gdT", "MAIT", "NKT"  # 非常规 T 也包含
]
adata_t = adata[
    adata.obs["cell_type_fine"].isin(t_types)
].copy()
print(f"T cells: {adata_t.n_obs}")

**输出：**

> 📊 输出：
> T cells: 20409

In [ ]:
# 重新计算 neighbor graph + diffusion map
sc.pp.neighbors(
    adata_t, use_rep="X_pca_harmony",
    n_neighbors=15, n_pcs=20
)
sc.tl.diffmap(adata_t, n_comps=10)

# Root: CD4 naive 中 CCR7 表达最高的细胞
naive_mask = (
    adata_t.obs["cell_type_fine"] == "CD4 naive"
)
ccr7_expr = adata_t[naive_mask, "CCR7"].X
if hasattr(ccr7_expr, "toarray"):
    ccr7_expr = ccr7_expr.toarray()
root_idx_t = np.where(naive_mask)[0][
    np.argmax(ccr7_expr.flatten())
]
adata_t.uns["iroot"] = root_idx_t

sc.tl.dpt(adata_t, n_dcs=10)

In [ ]:
# 各亚群的中位拟时序
median_dpt = (
    adata_t.obs
    .groupby("cell_type_fine")["dpt_pseudotime"]
    .median()
    .sort_values()
)
print(median_dpt.to_string())

**输出：**

> 📊 输出：
> cell_type_fine
> CD4 naive        0.1023
> CD4 memory       0.3245
> Treg             0.3891
> CD8 memory       0.5012
> MAIT             0.5634
> gdT              0.6127
> NKT              0.6543
> CD8 effector     0.7821

这个排列符合生物学预期：CD4 naive（最原始）→ CD4 memory → Treg → CD8 memory → CD8 effector（最分化）。MAIT 和 gdT 是非常规 T 细胞，在轨迹中间偏后——它们和经典 T 细胞共享部分转录程序但走了不同的分化路径。

In [ ]:
# 保存
tcell_dpt = adata_t.obs[[
    "cell_type_fine", "condition", "donor",
    "dpt_pseudotime"
]].copy()
tcell_dpt.to_csv("results/tcell_dpt.csv")
print("✅ T cell DPT saved")

图 4：T 细胞分化轨迹

💡 Tip：T 细胞轨迹的"分叉"问题

T 细胞的分化不是一条直线——它是一棵树。CD4 naive 可以分化成 CD4 memory 或 Treg；CD8 可以独立分化。DPT 只给每个细胞一个数值（一维），所以它会把这棵树"投影"到一条线上，导致分支信息丢失。

如果你需要建模分叉，可以尝试 Monocle 3 或 Palantir——它们能检测分支点。

## Part D：RNA Velocity（scVelo）——为什么我们跳过了

### 什么是 RNA velocity

RNA velocity（La Manno et al., Nature, 2018）利用未剪接（unspliced）和已剪接（spliced）mRNA 的比值来推断基因表达变化的"方向"。如果一个基因的 unspliced RNA 比预期多，说明这个基因正在被上调——反之则在下调。

这是目前唯一不需要先验知识就能推断分化方向的方法。

### 为什么我们不能用

In [ ]:
# 检查是否有 spliced/unspliced 信息
print("Layers in adata:", list(adata.layers.keys()))
has_spliced = "spliced" in adata.layers
has_unspliced = "unspliced" in adata.layers
print(f"Has spliced: {has_spliced}")
print(f"Has unspliced: {has_unspliced}")

**输出：**

> 📊 输出：
> Layers in adata: ['counts']
> Has spliced: False
> Has unspliced: False

⚠️ 踩坑预警：标准 10x CellRanger 输出不包含 spliced/unspliced 信息

scVelo 需要区分 spliced 和 unspliced 的 UMI 计数。标准的 CellRanger pipeline 不输出这些信息——你需要用 velocyto（velocyto run）或 STARsolo 重新比对原始 FASTQ 文件，生成 .loom 文件。

对于 GEO 上只提供 CellRanger 输出矩阵的数据集（如 GSE136103），无法直接做 RNA velocity。这是一个常见的遗憾——如果你自己的项目需要做 velocity，一定要在数据处理阶段就跑 velocyto。

### 替代方案

如果你有 spliced/unspliced 数据，推荐的工作流是：

velocyto run 处理 FASTQ → .loom 文件
合并 .loom 到 adata：scvelo.utils.merge(adata, ldata)
scvelo.pp.moments() → scvelo.tl.velocity() → scvelo.tl.velocity_graph()
scvelo.pl.velocity_embedding_stream() 生成箭头图

完整代码参见 GitHub 仓库的 analysis/10_trajectory.py（velocity 部分已注释）。

## Part E：CytoTRACE2——发育潜能评分

### 什么是 CytoTRACE2

CytoTRACE2（Gulati et al., Science, 2020 初版；2024 v2）通过基因表达多样性和特定基因集来预测每个细胞的"分化潜能"——潜能高的细胞更接近干细胞/祖细胞状态。

它和 DPT 的区别：DPT 需要你指定 root cell，CytoTRACE 不需要——它基于一个普遍的规律（未分化细胞的转录多样性更高）来自动判断。

### 安装问题

In [ ]:
try:
    import cytotrace2
    print("✅ CytoTRACE2 available")
except ImportError:
    print("❌ CytoTRACE2 not installed")
    print("   Install: pip install cytotrace2")
    print("   Skipping CytoTRACE2 analysis")

**输出：**

> 📊 输出：
> ❌ CytoTRACE2 not installed
>    Install: pip install cytotrace2
>    Skipping CytoTRACE2 analysis

💡 Tip：CytoTRACE2 的安装和使用

CytoTRACE2 目前需要从 GitHub 安装：pip install git+https://github.com/digitalcytometry/cytotrace2。它依赖 PyTorch，如果你的环境已经有 TensorFlow 可能会有冲突。建议在独立的 conda 环境中运行。

如果只是想快速评估细胞的"分化程度"，CytoTRACE v1（基于基因计数的简单方法）也是一个选择：pip install cytotrace。

## 📖 与原文比较

与 Ramachandran et al., 2019 对照：

原文的轨迹分析是他们最重要的结论之一：

Monocyte → SAMac 分化轨迹：原文使用 Monocle 和 RNA velocity 证明了外周血单核细胞浸润肝脏后分化为 TREM2+CD9+ SAMac。我们用 DPT 重现了这条轨迹——S100A8/S100A9 沿拟时序下降、TREM2/C1QA 上升，方向一致。

RNA velocity 箭头：原文展示了从单核细胞指向 SAMac 的 velocity 箭头。由于 GSE136103 的公开数据不包含 spliced/unspliced 信息，我们无法重现这一分析。但 DPT + 基因趋势分析提供了间接支持。

Kupffer 细胞的位置：在原文的轨迹中，Kupffer 细胞位于一个独立的分支——它们是肝脏驻留的巨噬细胞，不在 monocyte-to-SAMac 轨迹上。我们的 DPT 中 Kupffer 细胞的拟时序也确实和 CD14 Mono 有显著差距。

差异： 原文还做了 CytoTRACE 分析来验证单核细胞的高分化潜能。由于包安装问题我们未能重现，但这是一个值得后续补充的分析。

## 方法论反思

### 轨迹分析的可信度层级

不同方法提供不同程度的证据强度：

方法
证据强度
需要什么
能推断方向吗

PAGA
⭐⭐
聚类 + kNN 图
❌

DPT
⭐⭐⭐
选 root cell
需要先验

RNA velocity
⭐⭐⭐⭐
spliced/unspliced
✅

CytoTRACE
⭐⭐⭐
基因表达矩阵
部分

Lineage tracing
⭐⭐⭐⭐⭐
实验条形码
✅

在论文中，最可靠的做法是多种方法交叉验证。如果 DPT、RNA velocity 和 CytoTRACE 都指向同一个方向，那这条轨迹的可信度就很高。

### DPT 的局限性

单一维度：DPT 把细胞排在一条线上，不能处理分叉（一个祖细胞分化出两种命运）
对噪声敏感：单个 root cell 的选择可能引入偏差。解决方案：选多个候选 root cell，看结果是否稳健
假设连续性：DPT 假设分化是连续的渐变过程。如果存在"跳跃式"转变（如细胞重编程），DPT 可能捕捉不到

### 什么时候轨迹分析不适用

没有连续分化过程的数据：比如血液免疫细胞 + 上皮细胞的混合——它们之间没有分化关系
批次效应没有完全去除：残余的批次效应会被误认为"分化轨迹"
离散的细胞状态：如果细胞类型之间有清晰的边界（而不是连续过渡），强行拟合轨迹没有意义

---

### 🔬 方法选择指南

🔬 方法选择指南：轨迹分析方法全景对比
方法输入检测分支方向性速度最佳场景PAGA ✓KNN图 + 聚类拓扑连接图无⚡ 快全局概览（推荐首选）DPT ✓扩散图需预设root伪时间排序⚡ 快线性/简单分化轨迹Monocle 3UMAP + 图学习自动检测伪时间排序中复杂分支轨迹（R生态）Slingshot降维坐标 + MST自动检测伪时间排序快简单分支（R生态）scVelo剪接/未剪接RNA比值方向推断RNA velocity向量中需要剪接信息的方向推断Palantir扩散图 + 马尔可夫链概率性分支分化概率中概率性分支+多终点CytoTRACE2 ✓表达矩阵否发育潜能评分中评估干性/分化状态
我们的组合策略：PAGA + DPT + CytoTRACE2。理由：

① PAGA提供全局地图——先看清哪些细胞群之间有连接/分化关系，再决定做哪条轨迹；
② DPT量化伪时间——在PAGA确认的分化路径上（如单核→巨噬），DPT给出每个细胞的伪时间排序；
③ CytoTRACE2独立验证——它不依赖伪时间，而是从基因表达多样性评估发育潜能，可以交叉验证DPT的方向。

为什么我们跳过了 RNA Velocity

GSE136103数据使用10x Chromium v2化学（3'端测序），且GEO下载的是CellRanger输出的filtered matrix，不包含剪接/未剪接reads区分。RNA velocity需要从原始BAM文件用velocyto或STARsolo重新定量，这超出了本教程的范围。

如果你有velocity数据：推荐使用scVelo（dynamical mode）或更新的MultiVelo（整合RNA velocity + chromatin accessibility）。注意velocity结果高度依赖剪接定量的质量——Li et al. (2023) 指出，在许多数据集上velocity的方向推断并不可靠，需谨慎解读。

选择指南

• 第一步总是PAGA——快速、无参数，给你全局直觉
• 确认分化方向：如果有生物学先验（如HSC→progenitor→mature），用DPT；如果没有，用CytoTRACE2评估哪端更"干"
• 有复杂分支？ → Monocle 3或Palantir
• 有剪接信息？ → scVelo提供真正的方向性

---

## 小结

这一篇我们完成了：

✅ PAGA 全局拓扑分析（12 种大类的连接关系）
✅ Myeloid DPT（Monocyte → Macrophage 分化轨迹，root=S100A8 最高的 CD14 Mono）
✅ T cell DPT（CD4 naive → effector/memory 轨迹，root=CCR7 最高的 CD4 naive）
✅ scVelo 跳过（无 spliced/unspliced 数据）
✅ CytoTRACE2 跳过（未安装）
✅ 输出文件：paga_connectivity.csv、myeloid_dpt.csv、tcell_dpt.csv

关键发现：

PAGA 确认 Monocytes-Macrophages 强连接（connectivity=0.74）
髓系 DPT 重现了 monocyte-to-SAMac 轨迹：S100A8/S100A9 下降，C1QA/TREM2 上升
T 细胞 DPT 排序符合生物学预期：CD4 naive → memory → Treg → effector

核心教训： 拟时序是计算推断，不是实验测量。Root cell 的选择需要生物学先验。多种方法交叉验证是提高可信度的关键。在论文中，永远要用"suggests"而不是"proves"来描述轨迹分析的结论。

下一篇，我们将用共表达分析和基因集评分来量化每个细胞的"疾病状态"——这是从单基因层面走向通路/模块层面的最后一步。